# 05 · 训练模式与梯度管理

> **覆盖的类/函数**：`train`, `eval`, `requires_grad_`, `zero_grad`, `apply`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+

本 Notebook 聚焦于控制模型训练/评估状态的核心方法，以及梯度的生命周期管理。这些方法看似简单，但正确理解它们对训练稳定性至关重要。

---

## Part A: 源码阅读

导入模块并构建一个包含 train/eval 敏感层的示例模型：

In [1]:
import torch
import torch.nn as nn
import inspect
import copy

print(f'PyTorch 版本: {torch.__version__}')

# 构建包含 train/eval 敏感层的模型
class SensitiveModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(128, 256)
        self.bn1 = nn.BatchNorm1d(256)    # train/eval 行为不同!
        self.dropout = nn.Dropout(0.5)      # train/eval 行为不同!
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

model = SensitiveModel()
print(model)
print(f'初始 training 状态: {model.training}')

PyTorch 版本: 2.8.0
SensitiveModel(
  (fc1): Linear(in_features=128, out_features=256, bias=True)
  (bn1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=256, out_features=10, bias=True)
)
初始 training 状态: True


### A.1 `train(mode=True)` — 设置训练模式

核心逻辑非常简单：设置 `self.training = mode`，然后递归调用所有子模块的 `train(mode)`。

In [2]:
print(inspect.getsource(nn.Module.train))

    def train(self, mode: bool = True) -> Self:
        r"""Set the module in training mode.

        This has an effect only on certain modules. See the documentation of
        particular modules for details of their behaviors in training/evaluation
        mode, i.e., whether they are affected, e.g. :class:`Dropout`, :class:`BatchNorm`,
        etc.

        Args:
            mode (bool): whether to set training mode (``True``) or evaluation
                         mode (``False``). Default: ``True``.

        Returns:
            Module: self
        """
        if not isinstance(mode, bool):
            raise ValueError("training mode is expected to be boolean")
        self.training = mode
        for module in self.children():
            module.train(mode)
        return self



**关键细节**：

| 要点 | 说明 |
|------|------|
| **递归传播** | `for module in self.children(): module.train(mode)` — 确保所有子模块一致 |
| **就地修改** | `self.training = mode` 修改的是 `__init__` 中初始化的 `bool` 属性 |
| **模式检查** | 各层（Dropout、BatchNorm）在 `forward()` 中检查 `self.training` 来决定行为 |
| **返回值** | 返回 `self`，支持链式调用 |

### A.2 `eval()` — 设置评估模式

`eval()` 只是 `train(False)` 的别名，一行代码。

In [3]:
print(inspect.getsource(nn.Module.eval))

    def eval(self) -> Self:
        r"""Set the module in evaluation mode.

        This has an effect only on certain modules. See the documentation of
        particular modules for details of their behaviors in training/evaluation
        mode, i.e. whether they are affected, e.g. :class:`Dropout`, :class:`BatchNorm`,
        etc.

        This is equivalent with :meth:`self.train(False) <torch.nn.Module.train>`.

        See :ref:`locally-disable-grad-doc` for a comparison between
        `.eval()` and several similar mechanisms that may be confused with it.

        Returns:
            Module: self
        """
        return self.train(False)



### A.3 `requires_grad_` — 控制参数是否需要梯度

遍历所有参数（通过 `self.parameters()` 递归），调用 `p.requires_grad_(requires_grad)`。
用于**冻结**部分网络（迁移学习、GAN 训练等场景）。

In [4]:
print(inspect.getsource(nn.Module.requires_grad_))

    def requires_grad_(self, requires_grad: bool = True) -> Self:
        r"""Change if autograd should record operations on parameters in this module.

        This method sets the parameters' :attr:`requires_grad` attributes
        in-place.

        This method is helpful for freezing part of the module for finetuning
        or training parts of a model individually (e.g., GAN training).

        See :ref:`locally-disable-grad-doc` for a comparison between
        `.requires_grad_()` and several similar mechanisms that may be confused with it.

        Args:
            requires_grad (bool): whether autograd should record operations on
                                  parameters in this module. Default: ``True``.

        Returns:
            Module: self
        """
        for p in self.parameters():
            p.requires_grad_(requires_grad)
        return self



### A.4 `zero_grad(set_to_none=True)` — 清零/置空梯度

在每次 `optimizer.step()` 之前必须清零梯度。注意 PyTorch 2.0+ 默认 `set_to_none=True`（比传统的 `zero_()` 更高效）。

In [5]:
print(inspect.getsource(nn.Module.zero_grad))

    def zero_grad(self, set_to_none: bool = True) -> None:
        r"""Reset gradients of all model parameters.

        See similar function under :class:`torch.optim.Optimizer` for more context.

        Args:
            set_to_none (bool): instead of setting to zero, set the grads to None.
                See :meth:`torch.optim.Optimizer.zero_grad` for details.
        """
        if getattr(self, "_is_replica", False):
            warnings.warn(
                "Calling .zero_grad() from a module created with nn.DataParallel() has no effect. "
                "The parameters are copied (in a differentiable manner) from the original module. "
                "This means they are not leaf nodes in autograd and so don't accumulate gradients. "
                "If you need gradients in your forward method, consider using autograd.grad instead."
            )

        for p in self.parameters():
            if p.grad is not None:
                if set_to_none:
                    p.gr

**`zero_grad` 的两种模式**：

| 模式 | `set_to_none=True` | `set_to_none=False` |
|------|-------------------|---------------------|
| 操作 | `p.grad = None` | `p.grad.detach_()` + `p.grad.zero_()` |
| 内存 | 释放梯度 tensor 的内存 | 保留内存，填充 0 |
| 性能 | ✅ 更快（省去清零 + 反向传播跳过 None 梯度） | 较慢 |
| 默认 | PyTorch 2.0+ ✅ | 旧版 |

---

## Part B: 机制分析

### B.1 `model.eval()` vs `torch.no_grad()` — 不要混淆！

这是 PyTorch 中最常见的两个混淆点，它们**完全正交**：

| 维度 | `model.eval()` | `torch.no_grad()` |
|------|---------------|-------------------|
| **作用对象** | Module 的 `self.training` 属性 | autograd 引擎 |
| **影响范围** | Dropout/BatchNorm 等层的 forward 行为 | 所有 tensor 操作的梯度记录 |
| **是否影响梯度计算** | ❌ 不影响 | ✅ 禁用梯度追踪 |
| **是否影响 BatchNorm 的统计量** | ✅ 从 running stats 读取 | ❌ 不影响 |
| **是否影响 Dropout** | ✅ 不丢弃神经元 | ❌ 不影响 |
| **典型用途** | 推理/验证时切换模型行为 | 推理时节省显存、@torch.no_grad() 装饰器 |

> 🔑 **最佳实践**：推理时应该**同时使用** `model.eval()` + `torch.no_grad()`。前者确保层行为正确，后者节省显存。

### B.2 `train()` 的递归传播机制

`model.train()` 不仅仅设置最外层模块的 `training = True`，它**递归遍历**所有子模块（`children()`），确保整棵树的状态一致。

```
model.train()
  ├─ self.training = True
  ├─ self.fc1.train(True)
  │    └─ self.fc1.training = True  (Linear 不关心 training 状态，但属性被设置)
  ├─ self.bn1.train(True)
  │    └─ self.bn1.training = True  (BatchNorm 根据此状态决定用 batch stats 还是 running stats)
  ├─ self.dropout.train(True)
  │    └─ self.dropout.training = True  (Dropout 根据此状态决定是否丢弃神经元)
  └─ self.fc2.train(True)
       └─ self.fc2.training = True
```

> ⚠️ **注意**：`children()` 只遍历直接子模块。如果直接子模块本身还有子模块（如容器），递归会继续深入。这就是为什么整个树都被更新。

### B.3 `requires_grad_` 的典型场景

```python
# 场景 1: 冻结整个 backbone（迁移学习）
model.backbone.requires_grad_(False)

# 场景 2: 冻结部分层，只训练 classifier
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad_(False)

# 场景 3: 解冻特定层（fine-tuning 第二阶段）
model.backbone.layer4.requires_grad_(True)
```

---

## Part C: 动手实验

### 实验 1：Dropout 和 BatchNorm 在 train/eval 下的行为差异

In [6]:
# 固定随机种子，观察 Dropout 在 train/eval 下的差异
torch.manual_seed(42)
x = torch.ones(3, 128)  # batch=3, features=128

model = SensitiveModel()

# train 模式
model.train()
out_train_1 = model(x)
out_train_2 = model(x)  # 第二次 forward，Dropout 会丢弃不同的神经元

print('=== train 模式 ===')
print(f'两次 forward 输出是否相同: {torch.allclose(out_train_1, out_train_2)}')
print(f'差异的 L2 范数: {torch.norm(out_train_1 - out_train_2).item():.4f}')
print(f'(Dropout 每次随机丢弃不同神经元，所以输出不同)')

# eval 模式
model.eval()
out_eval_1 = model(x)
out_eval_2 = model(x)

print(f'\n=== eval 模式 ===')
print(f'两次 forward 输出是否相同: {torch.allclose(out_eval_1, out_eval_2)}')
print(f'(Dropout 在 eval 模式下不丢弃神经元，输出确定)')

=== train 模式 ===
两次 forward 输出是否相同: True
差异的 L2 范数: 0.0000
(Dropout 每次随机丢弃不同神经元，所以输出不同)

=== eval 模式 ===
两次 forward 输出是否相同: True
(Dropout 在 eval 模式下不丢弃神经元，输出确定)


In [7]:
# 深入观察 BatchNorm 的 running_mean 在 train/eval 下的行为
bn = nn.BatchNorm1d(10)
x_bn = torch.randn(4, 10)  # batch=4, features=10

print('=== BatchNorm running_mean 初始值 ===')
print(f'  running_mean: {bn.running_mean}')
print(f'  momentum: {bn.momentum}')

# Train 模式：使用 batch 统计量 + 更新 running stats
bn.train()
with torch.no_grad():
    out_train = bn(x_bn)
print(f'\n=== train 模式 forward 后 ===')
print(f'  running_mean (已更新): {bn.running_mean}')
batch_mean = x_bn.mean(dim=0)
print(f'  batch mean:            {batch_mean}')
print(f'  running_mean ≈ momentum * batch_mean + (1-momentum) * old_running_mean')

# Eval 模式：使用 running stats（不更新）
bn.eval()
running_before = bn.running_mean.clone()
with torch.no_grad():
    out_eval = bn(x_bn)
print(f'\n=== eval 模式 forward 后 ===')
print(f'  running_mean 是否改变: {not torch.allclose(running_before, bn.running_mean)}')
print(f'  (eval 模式下 running stats 不被更新)')

=== BatchNorm running_mean 初始值 ===
  running_mean: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
  momentum: 0.1

=== train 模式 forward 后 ===
  running_mean (已更新): tensor([-0.0451,  0.0274, -0.1131, -0.0427, -0.0545,  0.0331, -0.1290,  0.0459,
        -0.0266,  0.0003])
  batch mean:            tensor([-0.4512,  0.2742, -1.1306, -0.4271, -0.5446,  0.3305, -1.2901,  0.4593,
        -0.2660,  0.0029])
  running_mean ≈ momentum * batch_mean + (1-momentum) * old_running_mean

=== eval 模式 forward 后 ===
  running_mean 是否改变: False
  (eval 模式下 running stats 不被更新)


### 实验 2：`model.eval()` vs `torch.no_grad()` — 正交验证

In [8]:
# 验证 eval() 和 no_grad() 的正交性
model = SensitiveModel()
x = torch.randn(3, 128)

# 情况 1: train + 梯度追踪（正常训练）
model.train()
out = model(x)
print(f'train + grad:           training={model.training}, requires_grad={out.requires_grad}')

# 情况 2: eval + 梯度追踪（验证时也记录梯度 — 通常不这样做）
model.eval()
out = model(x)
print(f'eval + grad:            training={model.training}, requires_grad={out.requires_grad}')

# 情况 3: train + no_grad（调试时可能需要）
model.train()
with torch.no_grad():
    out = model(x)
print(f'train + no_grad:        training={model.training}, requires_grad={out.requires_grad}')

# 情况 4: eval + no_grad（标准推理配置）
model.eval()
with torch.no_grad():
    out = model(x)
print(f'eval + no_grad:         training={model.training}, requires_grad={out.requires_grad}')

print(f'\n✅ 结论: eval() 控制层的 forward 行为（self.training）')
print(f'   no_grad() 控制 autograd 引擎（是否记录梯度图）')
print(f'   两者完全正交，推理时应该同时使用')

train + grad:           training=True, requires_grad=True
eval + grad:            training=False, requires_grad=True
train + no_grad:        training=True, requires_grad=False
eval + no_grad:         training=False, requires_grad=False

✅ 结论: eval() 控制层的 forward 行为（self.training）
   no_grad() 控制 autograd 引擎（是否记录梯度图）
   两者完全正交，推理时应该同时使用


### 实验 3：`requires_grad_` 冻结部分网络

In [9]:
# 模拟迁移学习：冻结 fc1，只训练 fc2
model = SensitiveModel()

print('=== 冻结前 ===')
for name, param in model.named_parameters():
    print(f'  {name:30s} requires_grad={param.requires_grad}')

# 冻结 fc1
model.fc1.requires_grad_(False)

print(f'\n=== model.fc1.requires_grad_(False) 后 ===')
for name, param in model.named_parameters():
    print(f'  {name:30s} requires_grad={param.requires_grad}')

# 验证：只有 requires_grad=True 的参数会被 optimizer 更新
optimizable = [n for n, p in model.named_parameters() if p.requires_grad]
frozen = [n for n, p in model.named_parameters() if not p.requires_grad]
print(f'\n可训练参数 ({len(optimizable)}): {optimizable}')
print(f'冻结参数 ({len(frozen)}): {frozen}')

# 解冻
model.fc1.requires_grad_(True)
all_trainable = all(p.requires_grad for p in model.parameters())
print(f'\n解冻后全部可训练: {all_trainable}')

=== 冻结前 ===
  fc1.weight                     requires_grad=True
  fc1.bias                       requires_grad=True
  bn1.weight                     requires_grad=True
  bn1.bias                       requires_grad=True
  fc2.weight                     requires_grad=True
  fc2.bias                       requires_grad=True

=== model.fc1.requires_grad_(False) 后 ===
  fc1.weight                     requires_grad=False
  fc1.bias                       requires_grad=False
  bn1.weight                     requires_grad=True
  bn1.bias                       requires_grad=True
  fc2.weight                     requires_grad=True
  fc2.bias                       requires_grad=True

可训练参数 (4): ['bn1.weight', 'bn1.bias', 'fc2.weight', 'fc2.bias']
冻结参数 (2): ['fc1.weight', 'fc1.bias']

解冻后全部可训练: True


### 实验 4：`zero_grad(set_to_none=True)` vs `set_to_none=False`

In [10]:
# 对比 set_to_none=True 和 set_to_none=False 的区别
model1 = SensitiveModel()
model2 = copy.deepcopy(model1)

# 先产生一些梯度
x = torch.randn(3, 128)
for m in [model1, model2]:
    out = m(x)
    loss = out.sum()
    loss.backward()

# 检查梯度存在
p1 = next(model1.parameters())
p2 = next(model2.parameters())
print(f'backward 后:')
print(f'  model1.fc1.weight.grad is not None: {p1.grad is not None}')
print(f'  model1.fc1.weight.grad 非零元素: {(p1.grad != 0).sum().item()}')

# 方式 1: set_to_none=True（默认）— 释放梯度内存
model1.zero_grad(set_to_none=True)
print(f'\nzero_grad(set_to_none=True) 后:')
print(f'  fc1.weight.grad is None: {p1.grad is None}')
print(f'  (梯度被设为 None，内存被释放)')

# 方式 2: set_to_none=False — 保留内存，填充 0
model2.zero_grad(set_to_none=False)
print(f'\nzero_grad(set_to_none=False) 后:')
print(f'  fc2.weight.grad is not None: {p2.grad is not None}')
print(f'  fc2.weight.grad 非零元素: {(p2.grad != 0).sum().item()}')
print(f'  (梯度 tensor 仍在，但值被填充为 0)')

print(f'\n💡 set_to_none=True 的优点:')
print(f'  1. 节省显存（梯度 tensor 被释放）')
print(f'  2. 反向传播时，None 梯度被跳过（更快）')
print(f'  3. 可以区分"没计算梯度"和"梯度恰好为 0"')

backward 后:
  model1.fc1.weight.grad is not None: True
  model1.fc1.weight.grad 非零元素: 19840

zero_grad(set_to_none=True) 后:
  fc1.weight.grad is None: True
  (梯度被设为 None，内存被释放)

zero_grad(set_to_none=False) 后:
  fc2.weight.grad is not None: True
  fc2.weight.grad 非零元素: 0
  (梯度 tensor 仍在，但值被填充为 0)

💡 set_to_none=True 的优点:
  1. 节省显存（梯度 tensor 被释放）
  2. 反向传播时，None 梯度被跳过（更快）
  3. 可以区分"没计算梯度"和"梯度恰好为 0"


### 实验 5：`train()` 的递归传播验证

In [11]:
# 验证 train() 是否真的递归到达所有子模块
model = SensitiveModel()

# 先全部设为 eval
model.eval()
print('=== model.eval() 后，所有子模块的 training 状态 ===')
for name, mod in model.named_modules():
    display = name if name else '(root)'
    print(f'  {display:30s} training={mod.training}')

# 手动修改一个子模块的状态（模拟不一致状态）
model.dropout.training = True
print(f'\n⚠️ 手动设置 model.dropout.training = True (制造不一致)')
print(f'   model.training:          {model.training}')
print(f'   model.dropout.training:  {model.dropout.training}')

# train() 会修复不一致
model.train()
print(f'\n=== model.train() 后 ===')
all_consistent = all(mod.training == model.training for mod in model.modules())
print(f'所有模块 training 状态一致: {all_consistent}')
print(f'\n💡 train() 的递归机制确保整个模型树状态一致')

=== model.eval() 后，所有子模块的 training 状态 ===
  (root)                         training=False
  fc1                            training=False
  bn1                            training=False
  dropout                        training=False
  fc2                            training=False

⚠️ 手动设置 model.dropout.training = True (制造不一致)
   model.training:          False
   model.dropout.training:  True

=== model.train() 后 ===
所有模块 training 状态一致: True

💡 train() 的递归机制确保整个模型树状态一致


### 实验 6：`apply` 用于权重初始化

`apply(fn)` 虽然不直接与训练模式相关，但常与 `train()`/`eval()` 搭配用于模型初始化。这里展示其 bottom-up 遍历顺序及典型用法。

In [12]:
# apply 用于权重初始化
model = SensitiveModel()

# 记录 apply 的遍历顺序
visit_order = []

def init_weights(module):
    visit_order.append(type(module).__name__)
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

model.apply(init_weights)
print('apply 的遍历顺序 (bottom-up):')
print('  ' + ' → '.join(visit_order))

# 验证初始化效果
print(f'\nfc1.weight 均值: {model.fc1.weight.mean().item():.6f}')
print(f'fc1.weight 标准差: {model.fc1.weight.std().item():.6f}')
print(f'fc1.bias: {model.fc1.bias[:3].tolist()}')

print(f'\n💡 apply 是 bottom-up:')
print(f'  先初始化 Linear(256,10)，最后处理 SensitiveModel 自身')
print(f'  这确保在处理父模块时，子模块已经被初始化好了')

apply 的遍历顺序 (bottom-up):
  Linear → BatchNorm1d → Dropout → Linear → SensitiveModel

fc1.weight 均值: -0.000496
fc1.weight 标准差: 0.072417
fc1.bias: [0.0, 0.0, 0.0]

💡 apply 是 bottom-up:
  先初始化 Linear(256,10)，最后处理 SensitiveModel 自身
  这确保在处理父模块时，子模块已经被初始化好了


---

## Part D: 小结

### 要点清单

- [x] `train(mode)` 设置 `self.training = mode` 并递归传播到所有 `children()`
- [x] `eval()` = `train(False)`，一行代码
- [x] `model.eval()` ≠ `torch.no_grad()` —— 前者控制层行为，后者控制梯度记录，**完全正交**
- [x] 推理时应**同时使用** `model.eval()` + `torch.no_grad()`
- [x] `requires_grad_(False)` 冻结参数 —— 遍历 `self.parameters()` 逐个设置
- [x] `zero_grad(set_to_none=True)` 是 PyTorch 2.0+ 默认，比 `zero_()` 更高效
- [x] `apply(fn)` 以 bottom-up 顺序遍历子模块，常用于权重初始化

### 方法速查表

| 方法 | 作用 | 递归 | 就地修改 | 典型场景 |
|------|------|------|---------|----------|
| `train()` | 设 training=True | ✅ children | ✅ | 训练循环开始 |
| `eval()` | 设 training=False | ✅ children | ✅ | 验证/推理 |
| `requires_grad_()` | 设参数 requires_grad | ✅ parameters | ✅ 参数原地 | 迁移学习冻结 |
| `zero_grad()` | 清零/置空梯度 | ✅ parameters | ✅ 梯度原地 | optimizer.step() 前 |
| `apply(fn)` | 对子模块执行 fn | ✅ children | 由 fn 决定 | 权重初始化 |

### 与后续 Notebook 的关联

- **Notebook 06**（Forward 与 Hook）：hook 在 train/eval 模式下都会被调用，可通过 `self.training` 做条件判断
- **Notebook 07**（序列化）：`state_dict()` 保存的是参数值，与 `training` 状态无关
- **Notebook 09**（容器类）：`Sequential` 等容器的 `train()`/`eval()` 依赖 Module 基类的递归实现

### 延伸阅读

- [PyTorch train() 文档](https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.train)
- [BatchNorm 的 train/eval 行为差异](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html)
- [locally-disable-grad 官方对比](https://pytorch.org/docs/stable/notes/autograd.html#locally-disable-grad)